# 08 — MamaCare Kenya: Streamlit Application

**DSA 3900 | Final Year Project**  
**Author:** SelmahT · **Data:** KDHS 2022  

---

This notebook documents and launches the **MamaCare Kenya** Streamlit web application — 
a clinical decision-support prototype for maternal health risk stratification in Kenya.

## Design Philosophy

Unlike generic Streamlit templates (sidebar + sliders + output box), MamaCare Kenya was designed 
to resemble a real health product:

- **Role-gated access** — Clinician vs Researcher modes with different UI surfaces
- **Researcher data-use agreement** — ethical consent before accessing model internals
- **SHAP waterfall per patient** — explainability at the individual prediction level
- **Bayesian uncertainty tab** — goes beyond point-estimate leaderboards
- **Real Kenyan photography** — Unsplash imagery (free licence) instead of stock icons
- **Custom CSS design system** — teal (#0D6E6E) + cream (#F5F0E8), Inter + DM Serif Display

## Application Architecture

```
app.py
├── Home            → Role selection (Clinician / Researcher) + mission hero
├── Risk Assessment → 13-feature form → XGBoost predict → SHAP waterfall
├── Model Insights  → Leaderboard | Global SHAP | Bayesian HDI (researcher gate)
└── About           → KDHS, risk flags, pipeline, ethics, author
```

## Deployment Model Choice

**XGBoost** was selected (over TabTransformer and Stacking Ensemble) because:
1. Best individual performance: High-Risk F1 = 0.9985, AUC = 1.000
2. Native exact SHAP support via `TreeExplainer` — no kernel approximation
3. Sub-millisecond inference — suitable for real-time clinical interaction
4. Bayesian HDI [0.9909, 0.9995] at 94% credibility confirms deployment reliability

## Security & Ethics

| Measure | Implementation |
|---|---|
| No data persistence | All inputs processed in-memory; session state only |
| Role-based access | Clinician / Researcher gate on Home page |
| Researcher consent | Explicit data-use agreement before Model Insights |
| Clinical disclaimer | Persistent banner on all clinical pages |
| HTTPS | Handled automatically by Streamlit Cloud on deployment |

## Cell 1 — Install dependencies

Run once if not already installed.

In [1]:
# Cell 1 — Install / verify dependencies
import subprocess, sys

pkgs = ["streamlit", "shap", "xgboost", "scikit-learn", "joblib", "pandas", "matplotlib"]

for pkg in pkgs:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "show", pkg],
        capture_output=True, text=True
    )
    status = "✓" if result.returncode == 0 else "✗ MISSING"
    print(f"{status}  {pkg}")

print("\nRun: pip install streamlit shap  (if any marked ✗)")

✓  streamlit
✓  shap
✓  xgboost
✓  scikit-learn
✓  joblib
✓  pandas


Exception in thread Thread-15 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\tzin\AppData\Local\Python\pythoncore-3.14-64\Lib\threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "c:\Users\tzin\AppData\Local\Python\pythoncore-3.14-64\Lib\threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\tzin\AppData\Local\Python\pythoncore-3.14-64\Lib\subprocess.py", line 1613, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "c:\Users\tzin\AppData\Local\Python\pythoncore-3.14-64\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 27687: character maps to <undefined>


✓  matplotlib

Run: pip install streamlit shap  (if any marked ✗)


## Cell 2 — Verify model artefacts

All required files must exist in `03_Models_Saved/` before the app starts.

In [2]:
# Cell 2 — Check artefact availability
from pathlib import Path

ROOT  = Path("../../")
MOUT  = ROOT / "03_Models_Saved"
FOUT  = ROOT / "04_Outputs" / "figures"
ROUT  = ROOT / "04_Outputs" / "results"

required = [
    (MOUT / "scaler.pkl",           "StandardScaler"),
    (MOUT / "xgb_model.pkl",        "XGBoost model"),
    (MOUT / "shap_explainer.pkl",   "SHAP TreeExplainer"),
    (ROUT / "feature_map.csv",      "Feature name map"),
    (ROUT / "all_model_results.csv","All model results"),
    (FOUT / "shap_global_importance.png", "SHAP global chart"),
    (FOUT / "shap_beeswarm_high_risk.png","SHAP beeswarm"),
    (FOUT / "bayesian_posterior_f1.png",  "Bayesian posterior"),
]

all_ok = True
for path, label in required:
    ok = path.exists()
    status = "✓" if ok else "✗ MISSING"
    print(f"{status}  {label:<35} {path}")
    if not ok:
        all_ok = False

print()
if all_ok:
    print("All artefacts found ✓ — ready to launch app.")
else:
    print("⚠ Some artefacts missing. Run the relevant notebooks first.")

✓  StandardScaler                      ..\..\03_Models_Saved\scaler.pkl
✓  XGBoost model                       ..\..\03_Models_Saved\xgb_model.pkl
✓  SHAP TreeExplainer                  ..\..\03_Models_Saved\shap_explainer.pkl
✓  Feature name map                    ..\..\04_Outputs\results\feature_map.csv
✓  All model results                   ..\..\04_Outputs\results\all_model_results.csv
✓  SHAP global chart                   ..\..\04_Outputs\figures\shap_global_importance.png
✓  SHAP beeswarm                       ..\..\04_Outputs\figures\shap_beeswarm_high_risk.png
✓  Bayesian posterior                  ..\..\04_Outputs\figures\bayesian_posterior_f1.png

All artefacts found ✓ — ready to launch app.


## Cell 3 — Write app.py

Writes the complete Streamlit application to `05_App/app.py`.

> **Design decisions recorded here:**
> - 4-page structure: Home · Risk Assessment · Model Insights · About
> - Teal (`#0D6E6E`) + Cream (`#F5F0E8`) colour palette — WHO health dashboard feel
> - Inter + DM Serif Display typography
> - Role gate (Clinician / Researcher) at Home
> - Researcher data-use consent wall before Model Insights
> - SHAP waterfall computed live per patient using saved `shap_explainer.pkl`
> - No data persistence — in-memory session state only
> - Real Kenyan healthcare photography via Unsplash (free commercial licence)

## Cell 4 — Application source (display)

The complete `app.py` code is shown below for documentation purposes.
See `05_App/app.py` for the runnable file.

In [4]:
# Cell 4 — Display app source in notebook
app_path = Path("../../05_App/app.py")
if app_path.exists():
    print(app_path.read_text())
else:
    print("Run Cell 3 first to write app.py")

## Cell 5 — Launch the application

Runs `streamlit run app.py` in the background.
Open your browser at **http://localhost:8501** once this cell completes.

In [ ]:
# Cell 5 — Launch Streamlit app
import subprocess, sys, os, time
from pathlib import Path

app_path = Path("../../05_App/app.py").resolve()

print(f"Launching: streamlit run {app_path}")
print("Open → http://localhost:8501\n")

proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", str(app_path),
     "--server.headless", "true",
     "--server.port", "8501",
     "--theme.base", "light"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Print startup logs for ~5 seconds
import threading

def stream_output(p):
    for line in p.stdout:
        print(line, end="")

t = threading.Thread(target=stream_output, args=(proc,), daemon=True)
t.start()
time.sleep(5)

print("\n✅ Streamlit server started — visit http://localhost:8501")

Launching: streamlit run C:\Users\tzin\Desktop\maternal-health-risk-kenya\05_App\app.py
Open → http://localhost:8501


✅ Streamlit server started — visit http://localhost:8501


2026-10-31 22:42:26.925 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.100.14:8501
  External URL: http://41.90.185.248:8501

Exception in callback _ProactorBasePipeTransport._call_connection_lost()
handle: <Handle _ProactorBasePipeTransport._call_connection_lost()>
Traceback (most recent call last):
  File "c:\Users\tzin\AppData\Local\Python\pythoncore-3.14-64\Lib\asyncio\events.py", line 94, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\tzin\AppData\Local\Python\pythoncore-3.14-64\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
Exception in callback _ProactorBasePipeTransport._call_connection_los

## Cell 6 — Notebook Summary

Records the final application specification for the dissertation.

In [ ]:
# Cell 6 — Final summary
from pathlib import Path
import datetime

app_path = Path("../../05_App/app.py")

print("=" * 60)
print("  MamaCare Kenya — Application Summary")
print("=" * 60)
print(f"  App file       : {app_path.resolve()}")
print(f"  File size      : {app_path.stat().st_size / 1024:.1f} KB")
print(f"  Generated      : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print()
print("  Pages")
print("  ─────")
print("  1. Home           — Role gate + mission hero + key metrics")
print("  2. Risk Assessment — 13-feature form → XGBoost → SHAP waterfall")
print("  3. Model Insights — Leaderboard | Global SHAP | Bayesian HDI")
print("  4. About          — KDHS, risk flags, pipeline, ethics, author")
print()
print("  Security Measures")
print("  ─────────────────")
print("  ✓ Role-based access (Clinician / Researcher)")
print("  ✓ Researcher data-use agreement gate")
print("  ✓ No data persistence (in-memory session only)")
print("  ✓ Clinical disclaimer on all prediction pages")
print("  ✓ HTTPS via Streamlit Cloud on deployment")
print()
print("  Deployment Model : XGBoost")
print("  Explainability   : SHAP TreeExplainer (exact)")
print("  High-Risk F1     : 0.9985")
print("  AUC              : 1.0000")
print("  Bayesian HDI 94% : [0.9909, 0.9995]")
print()
print("  Notebook 08 complete ✓")

---

## Notebook 08 Complete

**MamaCare Kenya** is a fully functional Streamlit prototype demonstrating:

- End-to-end deployment of a trained XGBoost model with live SHAP explanations
- Role-based access control and researcher ethics gate
- Bayesian uncertainty communication beyond point-estimate leaderboards
- Professional health-dashboard visual design grounded in real Kenyan healthcare data

**Next steps for deployment:**
1. Push `05_App/` to GitHub
2. Connect repo to [Streamlit Cloud](https://streamlit.io/cloud)
3. Add `requirements.txt` to repo root
4. Set `PYTHONPATH` to repo root in Streamlit secrets if needed